# Lecture 1: Markov Decision Processes (MDP)
This notebook provides a hands-on implementation of **Markov Decision Processes (MDP)**, the fundamental mathematical framework underlying Reinforcement Learning.

---
**Topics:**
1. MDP Definition and Components
2. Policies and Value Functions
3. Bellman Equations
4. Value Iteration
5. Policy Iteration
6. Applications: Inventory Control and the Gambler's Problem


## 1.1 Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Libraries loaded successfully!")
print(f"NumPy version: {np.__version__}")

## 1.2 What is an MDP?

A **Markov Decision Process** is defined by the triple:

$$\mathcal{M} = (\mathcal{X}, \mathcal{A}, P_0)$$

| Symbol | Meaning |
|--------|---------|
| $\mathcal{X}$ | State space |
| $\mathcal{A}$ | Action space |
| $P_0(\cdot \mid x, a)$ | Transition kernel |
| $r(x, a)$ | Immediate reward function |
| $\gamma \in [0,1)$ | Discount factor |

**Dynamics:** At each step the agent observes state $X_t$, selects action $A_t$, and the system evolves as:
$$(X_{t+1}, R_{t+1}) \sim P_0(\cdot \mid X_t, A_t)$$

**Objective:** Maximise the total discounted reward:
$$R = \sum_{t=0}^{\infty} \gamma^t R_{t+1}$$


## 1.3 MDP Class Implementation

In [ ]:
class MDP:
    """
    Finite Markov Decision Process

    Parameters
    ----------
    states      : list of states
    actions     : list of actions
    transitions : {(s, a): {s': prob}} dictionary
    rewards     : {(s, a): float} reward function
    gamma       : discount factor
    """
    def __init__(self, states, actions, transitions, rewards, gamma=0.95):
        self.states = states
        self.actions = actions
        self.transitions = transitions
        self.rewards = rewards
        self.gamma = gamma
        self.n_states = len(states)
        self.n_actions = len(actions)
        self.state_idx = {s: i for i, s in enumerate(states)}
        self.action_idx = {a: i for i, a in enumerate(actions)}

    def step(self, state, action):
        """Simulate one transition: returns (next_state, reward)."""
        next_states = list(self.transitions[(state, action)].keys())
        probs = list(self.transitions[(state, action)].values())
        next_state = np.random.choice(next_states, p=probs)
        reward = self.rewards.get((state, action), 0.0)
        return next_state, reward

    def bellman_policy(self, V, policy):
        """Apply the Bellman operator for a fixed policy."""
        V_new = np.zeros(self.n_states)
        for i, s in enumerate(self.states):
            a = policy[s]
            V_new[i] = self.rewards.get((s, a), 0)
            for s_next, prob in self.transitions.get((s, a), {}).items():
                j = self.state_idx[s_next]
                V_new[i] += self.gamma * prob * V[j]
        return V_new

    def bellman_optimality(self, V):
        """Apply the Bellman optimality operator T*."""
        V_new = np.zeros(self.n_states)
        for i, s in enumerate(self.states):
            q_vals = []
            for a in self.actions:
                if (s, a) in self.transitions:
                    q = self.rewards.get((s, a), 0)
                    for s_next, prob in self.transitions[(s, a)].items():
                        j = self.state_idx[s_next]
                        q += self.gamma * prob * V[j]
                    q_vals.append(q)
            V_new[i] = max(q_vals) if q_vals else 0
        return V_new

print("MDP class defined ✓")

## 1.4 Example: GridWorld

A classic 4×4 grid-world environment.

```
 0  1  2  3
 4  5  6  7
 8  9 10 11
12 13 14 15(G)
```
- **Goal:** Reach cell 15 (reward +1)
- **Trap:** Cell 11 (reward −1, terminal)
- **Actions:** Up, Down, Left, Right
- **Slip probability:** 10% (agent moves sideways with small probability)


In [ ]:
def create_gridworld(grid_size=4, gamma=0.95, slip_prob=0.1):
    """Construct a 4x4 GridWorld MDP."""
    n = grid_size
    states = list(range(n * n))
    actions = ['up', 'down', 'left', 'right']
    goal_state = n * n - 1    # bottom-right corner
    trap_state = n * n - 5    # cell 11 for a 4x4 grid

    def apply_action(s, a):
        row, col = s // n, s % n
        if a == 'up'    and row > 0:   return s - n
        if a == 'down'  and row < n-1: return s + n
        if a == 'left'  and col > 0:   return s - 1
        if a == 'right' and col < n-1: return s + 1
        return s  # wall: stay in place

    transitions = {}
    rewards = {}

    for s in states:
        if s in [goal_state, trap_state]:   # terminal states
            for a in actions:
                transitions[(s, a)] = {s: 1.0}
                rewards[(s, a)] = 0.0
            continue

        for a in actions:
            main_next = apply_action(s, a)
            other_actions = [x for x in actions if x != a]
            trans = {main_next: 1 - slip_prob}
            for oa in other_actions:
                slip_next = apply_action(s, oa)
                trans[slip_next] = trans.get(slip_next, 0) + slip_prob / len(other_actions)
            transitions[(s, a)] = trans
            rewards[(s, a)] = (1.0  if main_next == goal_state else
                               -1.0 if main_next == trap_state else -0.01)

    mdp = MDP(states, actions, transitions, rewards, gamma)
    mdp.goal_state = goal_state
    mdp.trap_state = trap_state
    mdp.grid_size = n
    return mdp

gridworld = create_gridworld(gamma=0.95)
print(f"GridWorld created:")
print(f"  Number of states : {gridworld.n_states}")
print(f"  Number of actions: {gridworld.n_actions}")
print(f"  Discount factor  : {gridworld.gamma}")
print(f"  Goal state       : {gridworld.goal_state}")
print(f"  Trap state       : {gridworld.trap_state}")

## 1.5 Value Functions and Bellman Equations

### Policy Value Function
The value function under policy $\pi$:
$$V^\pi(x) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t R_{t+1} \mid X_0 = x\right]$$

**Bellman equation** for policy $\pi$:
$$V^\pi(x) = r(x, \pi(x)) + \gamma \sum_{y \in \mathcal{X}} P(x, \pi(x), y)\, V^\pi(y)$$

### Optimal Value Function
$$V^*(x) = \sup_{\pi} V^\pi(x)$$

**Bellman optimality equation:**
$$V^*(x) = \sup_{a \in \mathcal{A}} \left[ r(x,a) + \gamma \sum_{y \in \mathcal{X}} P(x,a,y)\, V^*(y) \right]$$


## 1.6 Value Iteration

In [ ]:
def value_iteration(mdp, tol=1e-6, max_iter=1000, verbose=True):
    """
    Value Iteration Algorithm

    Iterates  V_{k+1} = T* V_k  until convergence.
    By Banach's fixed-point theorem, convergence is geometric.
    """
    V = np.zeros(mdp.n_states)
    history = [V.copy()]
    errors = []

    for iteration in range(max_iter):
        V_new = mdp.bellman_optimality(V)
        error = np.max(np.abs(V_new - V))
        errors.append(error)
        V = V_new.copy()
        history.append(V.copy())

        if error < tol:
            if verbose:
                print(f"Converged! Iteration: {iteration+1}, Error: {error:.2e}")
            break

    # Extract greedy policy
    policy = {}
    Q = np.zeros((mdp.n_states, mdp.n_actions))
    for i, s in enumerate(mdp.states):
        for j, a in enumerate(mdp.actions):
            if (s, a) in mdp.transitions:
                Q[i, j] = mdp.rewards.get((s, a), 0)
                for s_next, prob in mdp.transitions[(s, a)].items():
                    k = mdp.state_idx[s_next]
                    Q[i, j] += mdp.gamma * prob * V[k]
        policy[s] = mdp.actions[np.argmax(Q[i])]

    return V, policy, errors, history

# Run value iteration
V_star, optimal_policy, errors, history = value_iteration(gridworld)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) Optimal value function
n = gridworld.grid_size
V_grid = V_star.reshape(n, n)
im = axes[0].imshow(V_grid, cmap='RdYlGn', aspect='auto')
axes[0].set_title('Optimal Value Function V*(s)', fontweight='bold')
for i in range(n):
    for j in range(n):
        axes[0].text(j, i, f'{V_grid[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[0])

# 2) Optimal policy
arrow_map = {'up': '\u2191', 'down': '\u2193', 'left': '\u2190', 'right': '\u2192'}
axes[1].set_xlim(-0.5, n-0.5)
axes[1].set_ylim(-0.5, n-0.5)
axes[1].set_title('Optimal Policy \u03c0*(s)', fontweight='bold')
for i in range(n):
    for j in range(n):
        s = i * n + j
        if s == gridworld.goal_state:
            axes[1].text(j, n-1-i, 'G\u2713', ha='center', va='center',
                        fontsize=14, color='green', fontweight='bold')
        elif s == gridworld.trap_state:
            axes[1].text(j, n-1-i, 'T\u2717', ha='center', va='center',
                        fontsize=14, color='red', fontweight='bold')
        else:
            axes[1].text(j, n-1-i, arrow_map[optimal_policy[s]],
                        ha='center', va='center', fontsize=18)
axes[1].set_xticks(range(n)); axes[1].set_yticks(range(n))
axes[1].grid(True, alpha=0.5)

# 3) Convergence curve
axes[2].semilogy(errors, 'b-o', markersize=4)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Max Error (log scale)')
axes[2].set_title('Value Iteration Convergence', fontweight='bold')
axes[2].axhline(y=1e-6, color='r', linestyle='--', label='Tolerance (1e-6)')
axes[2].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch1_value_iteration_en.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nOptimal policy (selected states):")
for s in [0, 1, 4, 5, 9, 10]:
    print(f"  State {s:2d}: {optimal_policy[s]}")

## 1.7 Policy Iteration

In [ ]:
def policy_evaluation(mdp, policy, tol=1e-8, max_iter=500):
    """
    Policy Evaluation: solves the Bellman linear system
    V^pi = r_pi + gamma * P_pi * V^pi
    """
    V = np.zeros(mdp.n_states)
    for _ in range(max_iter):
        V_new = mdp.bellman_policy(V, policy)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    return V

def policy_improvement(mdp, V):
    """
    Policy Improvement: derive the greedy policy from Q values
    pi_new(s) = argmax_a Q(s,a)
    """
    policy = {}
    for i, s in enumerate(mdp.states):
        best_val = -np.inf
        best_action = mdp.actions[0]
        for a in mdp.actions:
            if (s, a) not in mdp.transitions:
                continue
            q = mdp.rewards.get((s, a), 0)
            for s_next, prob in mdp.transitions[(s, a)].items():
                j = mdp.state_idx[s_next]
                q += mdp.gamma * prob * V[j]
            if q > best_val:
                best_val = q
                best_action = a
        policy[s] = best_action
    return policy

def policy_iteration(mdp, verbose=True):
    """
    Policy Iteration Algorithm:
    1. Evaluate current policy  ->  V^{pi_k}
    2. Improve policy           ->  pi_{k+1} = greedy(V^{pi_k})
    3. Repeat until no change
    """
    policy = {s: np.random.choice(mdp.actions) for s in mdp.states}
    n_iters = 0
    policy_changes = []

    while True:
        V = policy_evaluation(mdp, policy)
        new_policy = policy_improvement(mdp, V)
        changes = sum(1 for s in mdp.states if policy.get(s) != new_policy[s])
        policy_changes.append(changes)
        n_iters += 1
        if verbose:
            print(f"  Iteration {n_iters}: {changes} state(s) changed")
        if changes == 0:
            if verbose:
                print(f"Policy converged after {n_iters} iteration(s).")
            break
        policy = new_policy

    return V, policy, policy_changes

print("Running Policy Iteration...")
V_pi, optimal_policy_pi, policy_changes = policy_iteration(gridworld)

print(f"\nMax difference |V_VI - V_PI|: {np.max(np.abs(V_star - V_pi)):.2e}")
print("Policy Iteration converges to the same solution with fewer iterations!")

plt.figure(figsize=(8, 4))
plt.bar(range(1, len(policy_changes)+1), policy_changes, color='steelblue', alpha=0.8)
plt.xlabel('Policy Iteration Step')
plt.ylabel('Number of States Changed')
plt.title('Policy Iteration: States Changed per Step')
plt.xticks(range(1, len(policy_changes)+1))
plt.savefig('/mnt/user-data/outputs/ch1_policy_iteration_en.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.8 Application: Inventory Control

Implementation of **Example 1.1** from Szepesvári (Chapter 1).

Daily inventory management problem:
- $X_t$: inventory level at the end of day $t$
- $A_t$: order quantity
- $D_{t+1} \sim \text{Poisson}(\lambda)$: stochastic demand

$$X_{t+1} = ((X_t + A_t) \wedge M - D_{t+1})^+$$

**Reward:**
$$R_{t+1} = p \cdot \min(X_t + A_t, D_{t+1}) - c \cdot A_t - K \cdot \mathbf{1}[A_t > 0] - h \cdot X_{t+1}$$


In [ ]:
from scipy.stats import poisson

def create_inventory_mdp(M=10, lam=3, p=10, c=2, K=5, h=1, gamma=0.95):
    """
    Inventory Control MDP (Example 1.1 — Szepesvári)

    M   : maximum inventory capacity
    lam : Poisson demand mean
    p   : selling price per unit
    c   : per-unit ordering cost
    K   : fixed ordering cost
    h   : per-unit holding cost
    """
    states  = list(range(M + 1))
    actions = list(range(M + 1))

    max_demand = 3 * M
    demand_probs = [poisson.pmf(d, lam) for d in range(max_demand + 1)]
    total = sum(demand_probs)
    demand_probs = [p_ / total for p_ in demand_probs]

    transitions = {}
    rewards = {}

    for x in states:
        for a in actions:
            if x + a > M:
                continue
            trans = {}
            total_reward = 0.0
            for d, dp in enumerate(demand_probs):
                if dp < 1e-10:
                    continue
                x_after = min(x + a, M)
                x_next  = max(x_after - d, 0)
                sold    = min(x_after, d)
                order_cost   = (K + c * a) if a > 0 else 0
                holding_cost = h * x_next
                revenue      = p * sold
                r = revenue - order_cost - holding_cost
                trans[x_next] = trans.get(x_next, 0) + dp
                total_reward += dp * r
            transitions[(x, a)] = trans
            rewards[(x, a)]     = total_reward

    return MDP(states, actions, transitions, rewards, gamma)

inv_mdp = create_inventory_mdp(M=15, lam=4)
print(f"Inventory MDP: {inv_mdp.n_states} states, {inv_mdp.n_actions} actions")

print("\nSolving with value iteration...")
V_inv, policy_inv, errors_inv, _ = value_iteration(inv_mdp, verbose=False)

states_list = inv_mdp.states
opt_orders  = [int(policy_inv[x]) for x in states_list]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(states_list, opt_orders, color='darkorange', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Current Inventory Level (x)', fontsize=12)
axes[0].set_ylabel('Optimal Order Quantity A*(x)', fontsize=12)
axes[0].set_title('Optimal Inventory Policy\n(s,S)-type policy', fontweight='bold')
axes[0].axhline(y=0, color='black', linewidth=0.5)

order_threshold = max([x for x in states_list if policy_inv[x] > 0], default=0)
axes[0].axvline(x=order_threshold, color='red', linestyle='--',
                label=f'Reorder point s = {order_threshold}')
axes[0].legend()

axes[1].plot(states_list, V_inv, 'b-o', markersize=6, linewidth=2)
axes[1].set_xlabel('Inventory Level (x)', fontsize=12)
axes[1].set_ylabel('V*(x)', fontsize=12)
axes[1].set_title('Optimal Value Function\n(Expected Discounted Total Profit)', fontweight='bold')
axes[1].fill_between(states_list, V_inv, alpha=0.2)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch1_inventory_en.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nReorder point: order when inventory <= {order_threshold}")

## 1.9 Application: The Gambler's Problem (Example 1.2)

Example 1.2 from Szepesvári: the gambler's wealth evolves until it reaches $w^*$ or goes bankrupt.

$$X_{t+1} = (1 + S_{t+1} A_t) X_t, \quad S_{t+1} \in \{-1, +1\}$$

Objective: maximise $P(X_t \rightarrow w^*)$.


In [ ]:
def create_gambler_mdp(w_star=100, p_win=0.4, gamma=1.0):
    """
    Gambler's Problem MDP (Example 1.2 — Szepesvári)
    p_win: probability of winning a bet  (< 0.5: unfavourable game)
    """
    states = list(range(w_star + 1))
    transitions = {}
    rewards = {}

    for x in states:
        if x == 0 or x == w_star:
            for a in range(x + 1):
                transitions[(x, a)] = {x: 1.0}
                rewards[(x, a)] = 0.0
            continue

        max_stake = min(x, w_star - x)
        for a in range(1, max_stake + 1):
            win_state  = min(x + a, w_star)
            lose_state = max(x - a, 0)
            transitions[(x, a)] = {win_state: p_win, lose_state: 1 - p_win}
            rewards[(x, a)] = p_win if win_state == w_star else 0.0

    all_actions = list(range(1, w_star // 2 + 1))
    mdp = MDP(states, all_actions, transitions, rewards, gamma)
    return mdp, w_star

gambler_mdp, w_star = create_gambler_mdp(w_star=100, p_win=0.4)
print(f"Gambler MDP: {gambler_mdp.n_states} states")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for p_win, color, label in [(0.4, 'red',   'p=0.4 (unfavourable)'),
                              (0.5, 'blue',  'p=0.5 (fair)'),
                              (0.6, 'green', 'p=0.6 (favourable)')]:
    g_mdp, _ = create_gambler_mdp(w_star=100, p_win=p_win)
    V_g, pol_g, _, _ = value_iteration(g_mdp, tol=1e-9, verbose=False)
    axes[0].plot(g_mdp.states, V_g, color=color, linewidth=2, label=label)

axes[0].set_xlabel('Wealth (x)', fontsize=12)
axes[0].set_ylabel('V*(x) = P(reaching w*)', fontsize=12)
axes[0].set_title("Gambler's Problem: Optimal Winning Probability", fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0, 100)

g_mdp_04, _ = create_gambler_mdp(w_star=100, p_win=0.4)
V_g04, pol_g04, _, _ = value_iteration(g_mdp_04, tol=1e-9, verbose=False)
stakes = [pol_g04.get(x, 0) for x in g_mdp_04.states]

axes[1].bar(g_mdp_04.states[1:-1], stakes[1:-1], color='darkblue', alpha=0.7)
axes[1].set_xlabel('Wealth (x)', fontsize=12)
axes[1].set_ylabel('Optimal Stake A*(x)', fontsize=12)
axes[1].set_title('Optimal Betting Policy (p=0.4)', fontweight='bold')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch1_gambler_en.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gambler's problem solved!")

## 1.10 Summary

| Concept | Description |
|---------|-------------|
| **MDP** | Triple $(\mathcal{X}, \mathcal{A}, P_0)$ |
| **Value Function** | $V^\pi(x) = \mathbb{E}[\sum \gamma^t R_{t+1} \mid X_0=x]$ |
| **Bellman Equation** | $V^\pi = T^\pi V^\pi$ (linear system) |
| **Bellman Optimality** | $V^* = T^* V^*$ (nonlinear fixed point) |
| **Value Iteration** | $V_{k+1} = T^* V_k$, geometric convergence |
| **Policy Iteration** | Evaluation + improvement loop |

### Theoretical Guarantee (Banach Fixed-Point Theorem)
The operator $T^*$ is a contraction in the max-norm for $\gamma < 1$:
$$\|T^* V - T^* W\|_\infty \leq \gamma \|V - W\|_\infty$$

This guarantees that value iteration always converges to $V^*$.
